# 05 — Decision congelada, hold-out y Registry
Champion y challenger se eligen por CV de desarrollo ANTES del hold-out. Umbrales elegidos solo con predicciones OOF de desarrollo minimizando FP+3FN (hipotesis academica, no costo municipal). Sensibilidad para razones FN/FP 1,2,3,5,10: todos los umbrales fijados antes de consultar el hold-out.

La funcion conserva el resultado final y no vuelve a entrenar/optimizar al ejecutar Run All. La reproduccion posterior usa una sesion Python limpia y carga el alias, sin fit ni ajuste de umbral. Esto verifica el resultado ya calculado, no una nueva seleccion. No se cambia el ganador si el segundo tiene mejor AUC en hold-out.


In [1]:
from pathlib import Path
import sys, json
sys.dont_write_bytecode = True
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/cierre.py").is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from IPython.display import display
import pandas as pd


In [2]:
from src.cierre import freeze_selection, evaluate_final
frozen = freeze_selection()
print(json.dumps(frozen, indent=2))
final = evaluate_final()
display(pd.DataFrame({role: values['metrics'] for role, values in final['roles'].items()}).T)
print(json.dumps(final, indent=2))

{
  "identity": "f0bfa3d78673d26729138a0848fdfb3cb59f21999737077058db08610b287b52",
  "registry": "pset3-predial-urbano-modelo",
  "selection_metric": "development CV ROC-AUC; sd tie-break only",
  "classification_policy": "OOF-only minimize FP+3*FN; thresholds fixed before holdout",
  "cost_units": "academic relative units, not municipal money",
  "cost_ratios": [
    1,
    2,
    3,
    5,
    10
  ],
  "roles": {
    "champion": {
      "candidate": "logistica",
      "default_auc": 0.5704640008296623,
      "default_sd": 0.0022607293471367,
      "tuned_auc": 0.570509366805021,
      "tuned_sd": 0.0021992940850053266,
      "delta": 4.5365975358646615e-05,
      "improvement_conclusive": false,
      "run_id": "8ab70273357640a4b2a8e5719a448704",
      "config_key": "fc587fa83b160ae4d256bbcf646bd76d75d72f6fe6f40292ac0d20b39270d1ff",
      "parameters": {
        "model__C": 100.0,
        "model__class_weight": null
      },
      "gap_roc_auc_mean": 0.004523513476132483,
      "th

C:\Users\PC\Documents\Codex\2026-11-10\pset-2-verificar-conexi-n-con\pset_3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026/11/17 22:46:01 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at C:\Users\PC\Documents\Codex\2026-11-10\pset-2-verificar-conexi-n-con\pset_3\.venv\Lib\site-packages\mlflow\assistant\skills\mlflow-trace-instrumentation\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


,roc_auc,average_precision,balanced_accuracy,f1,accuracy,cost_per_obligation,tn,fp,fn,tp,precision,recall,flagged_fraction,fit_seconds,inference_seconds,batch_rows,inference_ms_per_row
champion,0.545967,0.638246,0.5,0.751171,0.6015,0.3985,0.0,797.0,0.0,1203.0,0.6015,1.0,1.0,0.045799,0.015891,2000.0,0.007946
challenger,0.548561,0.643605,0.5,0.751171,0.6015,0.3985,0.0,797.0,0.0,1203.0,0.6015,1.0,1.0,0.674797,0.013248,2000.0,0.006624


{
  "identity": "f0bfa3d78673d26729138a0848fdfb3cb59f21999737077058db08610b287b52",
  "registry": "pset3-predial-urbano-modelo",
  "roles": {
    "champion": {
      "candidate": "logistica",
      "threshold": 0.35999999999999993,
      "cv_auc_mean": 0.570509366805021,
      "cv_auc_sd": 0.0021992940850053266,
      "cv_gap_auc": 0.004523513476132483,
      "metrics": {
        "roc_auc": 0.5459667435343052,
        "average_precision": 0.6382462194735529,
        "balanced_accuracy": 0.5,
        "f1": 0.7511707773961911,
        "accuracy": 0.6015,
        "cost_per_obligation": 0.3985,
        "tn": 0,
        "fp": 797,
        "fn": 0,
        "tp": 1203,
        "precision": 0.6015,
        "recall": 1.0,
        "flagged_fraction": 1.0,
        "fit_seconds": 0.04579879995435476,
        "inference_seconds": 0.015891099930740893,
        "batch_rows": 2000,
        "inference_ms_per_row": 0.007945549965370446
      },
      "sensitivity": {
        "1": {
          "cost_per_o

In [3]:
import subprocess
check = subprocess.run([sys.executable, '-B', '-X', 'utf8', '-m', 'src.cierre', 'reproduce'], cwd=ROOT, capture_output=True, text=True, encoding='utf-8', check=True)
print(check.stdout)
from mlflow import MlflowClient
client = MlflowClient(tracking_uri='http://127.0.0.1:5001')
for alias in ['champion','challenger']:
    version = client.get_model_version_by_alias('pset3-predial-urbano-modelo', alias)
    print(alias, 'version', version.version, 'run_id', version.run_id)

{
  "registry_uri": "models:/pset3-predial-urbano-modelo@champion",
  "auc": 0.5459667435343052,
  "expected_auc": 0.5459667435343052,
  "exact_auc_match": true,
  "exact_predictions_match": true,
  "refit": false,
  "threshold_optimized_again": false
}

champion version 1 run_id 04f5fbb56cbd4e9d82d20fb5a4e10bc2
challenger version 2 run_id b9c485a40e7e4ea1832414b916163c82


## Interpretacion y limites
Revisar AUC, precision, recall, matriz, costo y proporcion de casos marcados. Un costo asimetrico alto puede favorecer marcar casi todas las obligaciones: no es una mejora operacional demostrada. El bootstrap pareado describe incertidumbre del hold-out, no reelige modelo/umbral. Logistica es mas explicable/compacta; MLP requiere mas parametros y es menos transparente. No hay eficacia municipal demostrada ni requisito Snowflake cumplido: la fuente real de esta ejecucion es Parquet sintetico local.